# Demo 08: Severity dataset-shift analysis

This deterministic, model-free analysis compares the V2 validation split with the held-out test split. It investigates whether differences in severity prevalence, category-conditioned severity, ticket length, wording, or duplicate coverage could help explain why severity accuracy is much higher on validation than on test. The results identify hypotheses to investigate; they do not prove a causal explanation for model errors.

## Imports and deterministic helper functions

The next cell imports only standard-library modules and Matplotlib, then defines reusable functions for validating the JSONL records, deriving transparent text features, and rendering tables. No model is loaded and no randomness is used.

In [ ]:
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt


SEVERITIES = ('P1', 'P2', 'P3', 'P4')
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)?")


def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing the project instructions."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Could not find the project root. Start JupyterLab from the repository root.')


def render_markdown_table(headers: list[str], rows: list[list[object]]) -> None:
    """Display a compact Markdown table with right-aligned numeric columns."""
    alignment = ['---'] + ['---:' for _ in headers[1:]]
    lines = [f"| {' | '.join(headers)} |", f"| {' | '.join(alignment)} |"]
    lines.extend(f"| {' | '.join(str(value) for value in row)} |" for row in rows)
    from IPython.display import Markdown, display
    display(Markdown('\n'.join(lines)))


def tokenize(text: str) -> list[str]:
    """Return lowercase alphanumeric tokens for deterministic lexical comparisons."""
    return TOKEN_PATTERN.findall(text.lower())


def extract_text_features(text: str) -> dict[str, int]:
    """Return simple, interpretable ticket-length features."""
    return {
        'characters': len(text),
        'words': len(tokenize(text)),
        'sentences': len([part for part in re.split(r'[.!?]+', text) if part.strip()]),
    }


## Load and validate both analysis splits

The next cell loads `validation_dataset_v2.jsonl` and `test_dataset.jsonl` directly from disk. It validates the message structure and expected severity labels, derives only in-memory features, and checks for exact duplicate ticket text across splits.

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
VALIDATION_FILE = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'datasets' / 'validation_dataset_v2.jsonl'
TEST_FILE = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'datasets' / 'test_dataset.jsonl'


def load_labeled_tickets(path: Path, split_name: str) -> list[dict[str, object]]:
    """Load one JSONL split and validate the fields needed for severity analysis."""
    if not path.is_file():
        raise FileNotFoundError(f'Missing required dataset: {path}')
    records = []
    with path.open(encoding='utf-8') as dataset_file:
        for line_number, line in enumerate(dataset_file, start=1):
            record = json.loads(line)
            messages = record.get('messages')
            if not isinstance(messages, list) or len(messages) != 2:
                raise ValueError(f'{split_name} line {line_number} must contain exactly two messages.')
            if [message.get('role') for message in messages] != ['user', 'assistant']:
                raise ValueError(f'{split_name} line {line_number} has an unexpected message order.')
            ticket = messages[0].get('content', '').strip()
            target = json.loads(messages[1].get('content', ''))
            if not ticket or not isinstance(target, dict) or set(target) != {'category', 'severity', 'summary'}:
                raise ValueError(f'{split_name} line {line_number} has an invalid ticket or target schema.')
            if target['severity'] not in SEVERITIES:
                raise ValueError(f'{split_name} line {line_number} has unsupported severity {target["severity"]!r}.')
            if not target['category'] or not target['summary']:
                raise ValueError(f'{split_name} line {line_number} has an empty target value.')
            records.append({'split': split_name, 'ticket': ticket, 'category': target['category'], 'severity': target['severity'], **extract_text_features(ticket)})
    if not records:
        raise ValueError(f'{split_name} is empty.')
    return records


validation_records = load_labeled_tickets(VALIDATION_FILE, 'Validation V2')
test_records = load_labeled_tickets(TEST_FILE, 'Test')
validation_tickets = {record['ticket'] for record in validation_records}
test_tickets = {record['ticket'] for record in test_records}
cross_split_duplicates = validation_tickets & test_tickets

render_markdown_table(
    ['Split', 'Records', 'Unique tickets', 'Exact cross-split duplicates'],
    [
        ['Validation V2', len(validation_records), len(validation_tickets), len(cross_split_duplicates)],
        ['Test', len(test_records), len(test_tickets), len(cross_split_duplicates)],
    ],
)


## Compare severity prevalence

The next cell compares label counts and percentages in a fixed P1-to-P4 order. It reports the absolute percentage-point difference and total-variation distance (TVD), a descriptive measure from 0 for identical distributions to 1 for completely different ones. The grouped bar chart makes any label imbalance immediately visible.

In [ ]:
def severity_distribution(records: list[dict[str, object]]) -> dict[str, float]:
    """Return severity proportions in the declared severity order."""
    counts = Counter(record['severity'] for record in records)
    return {severity: counts[severity] / len(records) for severity in SEVERITIES}


validation_distribution = severity_distribution(validation_records)
test_distribution = severity_distribution(test_records)
severity_rows = []
for severity in SEVERITIES:
    validation_count = sum(record['severity'] == severity for record in validation_records)
    test_count = sum(record['severity'] == severity for record in test_records)
    percentage_point_delta = 100 * (test_distribution[severity] - validation_distribution[severity])
    severity_rows.append([severity, validation_count, f"{validation_distribution[severity]:.1%}", test_count, f"{test_distribution[severity]:.1%}", f"{percentage_point_delta:+.1f} pp"])
severity_tvd = 0.5 * sum(abs(test_distribution[severity] - validation_distribution[severity]) for severity in SEVERITIES)
render_markdown_table(['Severity', 'Validation count', 'Validation share', 'Test count', 'Test share', 'Test - validation'], severity_rows)
print(f'Severity-distribution TVD: {severity_tvd:.3f}')

x_positions = range(len(SEVERITIES))
bar_width = 0.36
figure, axis = plt.subplots(figsize=(8, 4))
axis.bar([position - bar_width / 2 for position in x_positions], [validation_distribution[severity] for severity in SEVERITIES], width=bar_width, label='Validation V2')
axis.bar([position + bar_width / 2 for position in x_positions], [test_distribution[severity] for severity in SEVERITIES], width=bar_width, label='Test')
axis.set_xticks(list(x_positions), SEVERITIES)
axis.set_ylabel('Share of records')
axis.set_title('Severity-label distribution by split')
axis.yaxis.set_major_formatter('{x:.0%}')
axis.legend()
axis.grid(axis='y', alpha=0.25)
plt.show()


## Compare ticket structure and category-conditioned severity

The next cell checks whether the test tickets have a different length profile and whether categories map to different severities across the two splits. A heatmap shows the test-minus-validation change in `P(severity | category)`; this can reveal a shift hidden by the aggregate severity distribution.

In [ ]:
def mean(values: list[float]) -> float:
    """Return a deterministic arithmetic mean for a non-empty value list."""
    return sum(values) / len(values)


feature_rows = []
for split_name, records in [('Validation V2', validation_records), ('Test', test_records)]:
    for feature_name in ('characters', 'words', 'sentences'):
        values = [record[feature_name] for record in records]
        ordered_values = sorted(values)
        median = ordered_values[len(ordered_values) // 2]
        feature_rows.append([split_name, feature_name, f'{mean(values):.1f}', median, min(values), max(values)])
render_markdown_table(['Split', 'Feature', 'Mean', 'Median', 'Min', 'Max'], feature_rows)

figure, axis = plt.subplots(figsize=(8, 4))
axis.boxplot([[record['words'] for record in validation_records], [record['words'] for record in test_records]], tick_labels=['Validation V2', 'Test'], showfliers=True)
axis.set_ylabel('Words per ticket')
axis.set_title('Ticket-length distribution by split')
axis.grid(axis='y', alpha=0.25)
plt.show()

categories = sorted({record['category'] for record in validation_records + test_records})
def conditional_severity_distribution(records: list[dict[str, object]], category: str) -> dict[str, float]:
    category_records = [record for record in records if record['category'] == category]
    if not category_records:
        return {severity: 0.0 for severity in SEVERITIES}
    return severity_distribution(category_records)

difference_matrix = []
for category in categories:
    validation_conditional = conditional_severity_distribution(validation_records, category)
    test_conditional = conditional_severity_distribution(test_records, category)
    difference_matrix.append([test_conditional[severity] - validation_conditional[severity] for severity in SEVERITIES])

figure, axis = plt.subplots(figsize=(7, max(4, len(categories) * 0.42)))
image = axis.imshow(difference_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
axis.set_xticks(range(len(SEVERITIES)), SEVERITIES)
axis.set_yticks(range(len(categories)), categories)
axis.set_title('Change in P(severity | category): test minus validation')
figure.colorbar(image, ax=axis, label='Probability difference')
plt.show()


## Compare severity-specific wording

The next cell extracts lowercase word tokens and, separately for each severity and split, lists the most frequent terms. It also computes Jaccard overlap of the top terms between validation and test. Low overlap is evidence that the model may be seeing different severity cues at test time, even when label shares are similar.

In [ ]:
TOP_TERMS = 12


def top_terms(records: list[dict[str, object]], severity: str, limit: int) -> list[str]:
    """Return the most frequent lexical items for one severity with stable tie-breaking."""
    token_counts = Counter()
    for record in records:
        if record['severity'] == severity:
            token_counts.update(tokenize(record['ticket']))
    return [token for token, _ in sorted(token_counts.items(), key=lambda item: (-item[1], item[0]))[:limit]]


lexical_rows = []
lexical_overlap = {}
for severity in SEVERITIES:
    validation_terms = top_terms(validation_records, severity, TOP_TERMS)
    test_terms = top_terms(test_records, severity, TOP_TERMS)
    union = set(validation_terms) | set(test_terms)
    overlap = len(set(validation_terms) & set(test_terms)) / len(union) if union else 1.0
    lexical_overlap[severity] = overlap
    lexical_rows.append([severity, ', '.join(validation_terms), ', '.join(test_terms), f'{overlap:.1%}'])
render_markdown_table(['Severity', 'Validation V2 top terms', 'Test top terms', 'Top-term Jaccard overlap'], lexical_rows)


## Produce a diagnostic summary

The next cell turns the descriptive measurements into an explicit, reproducible checklist. It reports the largest observed shifts without claiming they caused the accuracy drop; use the findings to inspect examples, revise synthetic-data generation, or design a targeted retraining experiment.

In [ ]:
largest_severity_shift = max(SEVERITIES, key=lambda severity: abs(test_distribution[severity] - validation_distribution[severity]))
largest_category_shift = max(
    ((abs(delta), categories[row_index], SEVERITIES[column_index], delta) for row_index, row in enumerate(difference_matrix) for column_index, delta in enumerate(row)),
    key=lambda item: (item[0], item[1], item[2]),
)
mean_lexical_overlap = mean(list(lexical_overlap.values()))

diagnostic_rows = [
    ['Exact duplicate ticket texts', len(cross_split_duplicates), 'Duplicates can inflate validation results; zero supports a genuinely separate test set.'],
    ['Severity-distribution TVD', f'{severity_tvd:.3f}', 'Higher values indicate stronger aggregate label-prevalence shift.'],
    ['Largest severity share shift', f"{largest_severity_shift}: {100 * (test_distribution[largest_severity_shift] - validation_distribution[largest_severity_shift]):+.1f} pp", 'A label is more or less common in test than in validation.'],
    ['Largest category-conditioned shift', f"{largest_category_shift[1]} / {largest_category_shift[2]}: {100 * largest_category_shift[3]:+.1f} pp", 'The category-to-severity relationship differs between splits.'],
    ['Mean top-term overlap', f'{mean_lexical_overlap:.1%}', 'Lower overlap indicates different prominent lexical cues by severity.'],
]
render_markdown_table(['Diagnostic', 'Observed value', 'Interpretation'], diagnostic_rows)
print('Conclusion: investigate the largest shifts above before attributing the test severity drop solely to model overfitting.')
